<a href="https://colab.research.google.com/github/umemon254/Fly-rank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umemon254/Fly-rank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

SyntaxError: invalid syntax (450418994.py, line 1)

One row represents the daily performance of one content item for one pseudonymized client on one report date.

For development, I use March 2026. I chose a mid-panel month instead of the final month because June 2026 is the sealed final outcome period.

My prediction task is binary classification: predict whether a content item should be considered declining based on information available before the decision moment.

I exclude client_id and content_id from model features because they are identifiers, not meaningful predictive signals.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Features: Search and engagement metrics that are available at the decision moment.

Label/proxy: A declining-content outcome created separately from the future trend. The label itself and the columns used to create it are never used as features.

Context: report_date, client_id, and content_id. These fields are used for filtering, grouping, and validation, not for model training.

Excluded: IDs, future outcome information, and unavailable GA4 rows. IDs could let the model memorize patterns, future information would cause leakage, and GA4 zeros are not reliable when ga4_data_available is FALSE.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face token loaded:", HF_TOKEN is not None)

Hugging Face token loaded: True


In [5]:
!pip -q install duckdb pyarrow huggingface_hub

In [6]:
import duckdb
import pandas as pd
import os

print("Libraries loaded successfully")

Libraries loaded successfully


In [7]:
con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB is ready")

DuckDB is ready


In [8]:
con.execute(f"""
CREATE SECRET hf (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face access configured successfully")

Hugging Face access configured successfully


In [9]:
result = con.execute("""
SELECT COUNT(*) AS total_clients
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
)
""").df()

result

,total_clients
0,104


In [11]:
grain_check = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [12]:
count_window = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

count_window

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [13]:
availability_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS available_percent
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,available_percent
0,9841378,413966,4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This data has some important limitations. Client history does not start at the same time for every client, so the available history is unbalanced. Some early rows contain zero-filled GA4 metrics when GA4 data was not yet available, so those zeros do not mean zero engagement. The data can support directional and decision-support analysis, but it cannot prove that a content item declined because of one specific cause.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.